# Qwen3-1.7B SFT — Causal vs Noncausal vs Base — Colab Inference Notebook\n\nSelf-contained 3-way comparison for the two **SFT LoRA adapters** trained on GSM8K reasoning.\n\n## ⚠️ Known model accuracy\n\nThis is a **research prototype on a 1.7B model** — do not expect strong results.\n\n| Condition | GSM8K accuracy (full 1319-question eval) | Avg tokens |\n|-----------|------------------------------------------|------------|\n| Base Qwen3-1.7B | ~60–65% (estimated) | ~2000 (hits limit) |\n| **SFT Causal (this model)** | **77.3%** | **169** |\n| SFT Noncausal | ~78–80% (estimated) | ~300 |\n\nThe causal model trades a small accuracy drop vs noncausal for a large token reduction — that is the core research finding. On this 5-example test, results will vary and may not reflect the full-eval numbers above.\n\n## Models being compared\n\n| Model | Training data | Expected behaviour |\n|-------|--------------|--------------------|\n| **Base** | none | verbose, lower accuracy |\n| **SFT Noncausal** | full CoT chains | moderate accuracy, moderate tokens |\n| **SFT Causal** | PNS-pruned chains (causally necessary steps only) | similar accuracy, fewest tokens |\n\n## Loading strategy\n\n1. `gdown` downloads **both** adapter folders from Google Drive.\n2. Base `Qwen/Qwen3-1.7B` loaded once in **4-bit** (~0.9 GB VRAM).\n3. Causal adapter applied → run 5 examples → `disable_adapter()` for base run.\n4. Noncausal adapter loaded, old adapter freed → run 5 examples.\n5. 3-way comparison table printed.\n\n## Runtime\n\n**Runtime → Change runtime type → T4 GPU**. All three passes fit on T4 (16 GB) with 4-bit loading.

## 1 · Install dependencies

In [ ]:
!pip install -q --upgrade transformers accelerate peft bitsandbytes sentencepiece gdown

## 2 · Download both SFT adapters from Google Drive

Folder IDs are already set from the shared Drive links.

In [ ]:
import os, json, glob
import gdown

# ── Drive folder IDs ──────────────────────────────────────────────────────────
CAUSAL_FOLDER_ID    = "1Qytyim3tZF90jI07J3oRat120DD70QQW"
NONCAUSAL_FOLDER_ID = "1ydRx8RG2e4Uw98AXfHJUkTepUtyTbs3D"

CAUSAL_ROOT    = "/content/sft_causal"
NONCAUSAL_ROOT = "/content/sft_noncausal"


def download_adapter(folder_id, dest):
    print(f"\nDownloading adapter from Drive folder: {folder_id}  →  {dest}")
    os.makedirs(dest, exist_ok=True)
    gdown.download_folder(
        f"https://drive.google.com/drive/folders/{folder_id}",
        output=dest,
        quiet=False,
        use_cookies=False,
    )


def locate_adapter(root):
    """Find the adapter_config.json that is NOT inside a checkpoint subdir."""
    hits = glob.glob(os.path.join(root, "**", "adapter_config.json"), recursive=True)
    root_hits = [h for h in hits if "checkpoint-" not in h]
    assert root_hits, (
        f"adapter_config.json not found under {root}.\n"
        f"Files: {[f for _, _, fs in os.walk(root) for f in fs]}"
    )
    return os.path.dirname(root_hits[0])


download_adapter(CAUSAL_FOLDER_ID,    CAUSAL_ROOT)
download_adapter(NONCAUSAL_FOLDER_ID, NONCAUSAL_ROOT)

CAUSAL_ADAPTER_PATH    = locate_adapter(CAUSAL_ROOT)
NONCAUSAL_ADAPTER_PATH = locate_adapter(NONCAUSAL_ROOT)

# Optional sample test data bundled in either folder
sample_hits = (
    glob.glob(os.path.join(CAUSAL_ROOT,    "**", "sample_test.jsonl"), recursive=True) +
    glob.glob(os.path.join(NONCAUSAL_ROOT, "**", "sample_test.jsonl"), recursive=True)
)
SAMPLE_DATA_PATH = sample_hits[0] if sample_hits else None

for label, path in [("Causal adapter", CAUSAL_ADAPTER_PATH), ("Noncausal adapter", NONCAUSAL_ADAPTER_PATH)]:
    print(f"\n{label}: {path}")
    print(f"  Contents: {sorted(os.listdir(path))}")
print(f"\nSample data: {SAMPLE_DATA_PATH or '(not in Drive — hardcoded samples will be used)'}")

## 3 · Read adapter configs

In [ ]:
with open(os.path.join(CAUSAL_ADAPTER_PATH,    "adapter_config.json")) as f:
    CAUSAL_CFG = json.load(f)
with open(os.path.join(NONCAUSAL_ADAPTER_PATH, "adapter_config.json")) as f:
    NONCAUSAL_CFG = json.load(f)

BASE_MODEL_NAME = CAUSAL_CFG["base_model_name_or_path"]
assert BASE_MODEL_NAME == NONCAUSAL_CFG["base_model_name_or_path"], \
    "Causal and noncausal adapters must share the same base model!"

print(f"Base model : {BASE_MODEL_NAME}")
print(f"\nCausal adapter LoRA config:")
for k in ("peft_type", "r", "lora_alpha", "lora_dropout", "target_modules", "task_type"):
    if k in CAUSAL_CFG:
        print(f"  {k}: {CAUSAL_CFG[k]}")

# Read run_config.json if present (has training loss, n_train, etc.)
for label, path in [("causal", CAUSAL_ADAPTER_PATH), ("noncausal", NONCAUSAL_ADAPTER_PATH)]:
    rc_path = os.path.join(path, "run_config.json")
    if os.path.exists(rc_path):
        with open(rc_path) as f:
            rc = json.load(f)
        print(f"\n{label} run_config: n_train={rc.get('n_train')}, "
              f"epochs={rc.get('epochs')}, train_loss={rc.get('train_loss', 0):.4f}")

# Chat template
CHAT_TEMPLATE = None
tmpl = os.path.join(CAUSAL_ADAPTER_PATH, "chat_template.jinja")
if os.path.exists(tmpl):
    with open(tmpl) as f:
        CHAT_TEMPLATE = f.read()
    print(f"\nchat_template.jinja found ({len(CHAT_TEMPLATE)} chars) — will override tokenizer default.")

## 4 · Imports & 4-bit quantization config

In [ ]:
import re, time, gc
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

assert torch.cuda.is_available(), "4-bit loading via bitsandbytes requires a CUDA GPU. Switch runtime to GPU."

COMPUTE_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

BNB_CONFIG = BitsAndBytesConfig(
    load_in_4bit              = True,
    bnb_4bit_quant_type       = "nf4",
    bnb_4bit_use_double_quant = True,
    bnb_4bit_compute_dtype    = COMPUTE_DTYPE,
)

# Capped at 512 for the test run — 5 easy examples need at most ~150 tokens each.
# This prevents Colab from OOMing on unexpectedly long base-model generations.
MAX_NEW_TOKENS = {"gsm8k": 512, "math500": 1024, "external": 512}

print(f"GPU: {torch.cuda.get_device_name(0)} | compute dtype: {COMPUTE_DTYPE}")
print(f"max_new_tokens cap: {MAX_NEW_TOKENS}")

## 5 · Answer-extraction helpers

Copied from `algo/equivalent_ans.py`. Local-only grading — no LLM judge.

In [ ]:
def _extract_boxed(text: str) -> str:
    results, start = [], 0
    while True:
        idx = text.find(r"\boxed{", start)
        if idx == -1:
            break
        depth = 0
        for i in range(idx + 7, len(text)):
            if text[i] == "{":
                depth += 1
            elif text[i] == "}":
                if depth == 0:
                    results.append(text[idx + 7:i])
                    start = i + 1
                    break
                depth -= 1
        else:
            break
    return results[-1].strip() if results else ""


def _normalize(text: str) -> str:
    t = text.strip()
    t = re.sub(r"\\left|\\right|\\,|\\!", "", t)
    t = re.sub(r"\^\\circ|\\circ|\\degree|°", "", t)
    t = re.sub(r"\\text\{([^}]*)\}", r"\1", t)
    t = re.sub(r"\\dfrac", r"\\frac", t)
    t = re.sub(r"\\tfrac", r"\\frac", t)
    t = re.sub(r"\$+", "", t)
    t = re.sub(r"\s+", "", t)
    return t.lower()


def is_correct_local(extracted: str, ground_truth: str) -> bool:
    if not extracted:
        return False
    if _normalize(extracted) == _normalize(ground_truth):
        return True
    try:
        return float(extracted.replace(",", "").strip()) == float(str(ground_truth).replace(",", "").strip())
    except (ValueError, TypeError):
        return False


def extract_answer(text: str, dataset: str) -> str:
    boxed = _extract_boxed(text)
    if boxed:
        return boxed
    if dataset == "gsm8k":
        m = re.search(r"####\s*([\d,.\-]+)", text)
        if m:
            return m.group(1).replace(",", "").strip()
    return ""


def strip_think(text: str) -> str:
    return text.split("</think>", 1)[1].strip() if "</think>" in text else text


def count_steps(text: str) -> int:
    if "<think>" in text and "</think>" in text:
        think = text.split("<think>", 1)[1].split("</think>", 1)[0]
        return len([p for p in re.split(r"\n\n+", think) if p.strip()])
    return len([p for p in re.split(r"\n\n+", text) if p.strip()])


print("Helpers loaded.")

## 6 · Tokenizer + prompt builder

Loaded from the **causal adapter folder** (both adapters share the same base tokenizer).  
`chat_template.jinja` is applied if present — guarantees the exact template used during training.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(CAUSAL_ADAPTER_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"  # left-pad for generation — matches eval_correct.py

if CHAT_TEMPLATE:
    tokenizer.chat_template = CHAT_TEMPLATE
    print("Chat template overridden from chat_template.jinja")


def build_prompt(tok, question: str) -> str:
    """Mirrors build_prompt() in sft/eval_correct.py."""
    msgs   = [{"role": "user", "content": question.strip()}]
    kwargs = dict(tokenize=False, add_generation_prompt=True)
    try:
        return tok.apply_chat_template(msgs, enable_thinking=True, **kwargs)
    except TypeError:
        return tok.apply_chat_template(msgs, **kwargs)


_sample = build_prompt(tokenizer, "What is 2 + 2?")
print("-- sample prompt --")
print(_sample[:300])
print(f"\nprompt length: {len(_sample)} chars")

## 7 · Load base model (4-bit) + apply causal SFT adapter

`Qwen/Qwen3-1.7B` is loaded once from HuggingFace.  
The causal LoRA adapter is applied on top — same pattern as `sft/eval_correct.py`.

In [ ]:
print(f"Loading base model: {BASE_MODEL_NAME}")
_base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    quantization_config=BNB_CONFIG,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="eager",
)
print(f"  VRAM after base load: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

print(f"Applying causal SFT adapter: {CAUSAL_ADAPTER_PATH}")
causal_model = PeftModel.from_pretrained(_base, CAUSAL_ADAPTER_PATH)
causal_model.eval()
print(f"  VRAM after causal adapter: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

## 8 · Test examples\n\n**5 single-step or 2-step GSM8K problems** — kept as short as possible so all three model passes stay well under the 512-token cap and Colab does not OOM.\n\nThis is a sanity check, not a benchmark. Known full-eval accuracy (1319 questions): SFT Causal ~77%, SFT Noncausal ~78–80%. Expect 3–4 correct out of 5 for both adapted models; the base will likely score lower and generate far more tokens.

In [ ]:
_HARDCODED_SAMPLES = [
    # 1 — single multiplication
    {"dataset": "gsm8k", "answer": "540",
     "question": "James decides to run 3 sprints 3 times a week. "
                 "He runs 60 meters each sprint. "
                 "How many total meters does he run a week?"},
    # 2 — subtract then multiply (2 steps)
    {"dataset": "gsm8k", "answer": "18",
     "question": "Janet's ducks lay 16 eggs per day. She eats three for breakfast "
                 "every morning and bakes muffins for her friends every day with four. "
                 "She sells the remainder at the farmers' market daily for $2 per fresh "
                 "duck egg. How much in dollars does she make every day at the farmers' market?"},
    # 3 — rate × fraction of hour (2 steps)
    {"dataset": "gsm8k", "answer": "10",
     "question": "Weng earns $12 an hour for babysitting. "
                 "Yesterday, she just did 50 minutes of babysitting. "
                 "How much did she earn?"},
    # 4 — pages left after two days, halve (2 steps)
    {"dataset": "gsm8k", "answer": "42",
     "question": "Julie is reading a 120-page book. Yesterday, she was able to read "
                 "12 pages and today, she read twice as many pages as yesterday. "
                 "If she wants to read half of the remaining pages tomorrow, "
                 "how many pages should she read tomorrow?"},
    # 5 — percentage down payment (2 steps, from iter_2 training data)
    {"dataset": "gsm8k", "answer": "56000",
     "question": "Roger bought a house for $100,000. He was able to pay 20% down, "
                 "and his parents paid off an additional 30% of the remaining balance. "
                 "How much money does Roger still owe on his house?"},
]

if SAMPLE_DATA_PATH:
    with open(SAMPLE_DATA_PATH, encoding="utf-8") as f:
        SAMPLES = [json.loads(line) for line in f if line.strip()]
    print(f"Loaded {len(SAMPLES)} samples from {SAMPLE_DATA_PATH}")
else:
    SAMPLES = _HARDCODED_SAMPLES
    print(f"Using {len(SAMPLES)} hardcoded samples")

print(f"\nReminder: full-eval accuracy is SFT Causal ~77.3%, SFT Noncausal ~78-80%.")
print(f"On 5 easy examples the numbers will fluctuate — treat this as a smoke test only.\n")
for i, s in enumerate(SAMPLES, 1):
    print(f"  [{i}] ans={s['answer']!r:8s}  {s['question'][:70]!r}")

## 9 · Generation + display helpers

In [ ]:
@torch.no_grad()
def generate_one(model, tok, question: str, dataset: str) -> dict:
    prompt = build_prompt(tok, question)
    enc = tok(prompt, return_tensors="pt", truncation=True, max_length=1024)
    enc = {k: v.to(model.device) for k, v in enc.items()}

    max_new = MAX_NEW_TOKENS.get(dataset, MAX_NEW_TOKENS["external"])
    t0 = time.time()
    out = model.generate(
        **enc,
        max_new_tokens=max_new,
        do_sample=False,
        temperature=1.0,
        top_p=1.0,
        pad_token_id=tok.eos_token_id,
    )
    in_len   = enc["input_ids"].shape[1]
    new_toks = out[0][in_len:]
    response = tok.decode(new_toks, skip_special_tokens=True)

    return {
        "response":    response,
        "extracted":   extract_answer(response, dataset),
        "answer_tail": strip_think(response),
        "new_tokens":  int(len(new_toks)),
        "step_count":  count_steps(response),
        "seconds":     time.time() - t0,
    }


def show_result(idx, ex, result, model_label):
    sep = "=" * 78
    print(sep)
    print(f"#{idx:02d}  [{ex.get('dataset','external')}]  model: {model_label}")
    print(sep)
    print("QUESTION:")
    q = ex["question"]
    print(q[:600] + ("..." if len(q) > 600 else ""))
    print()
    expected = ex.get("answer", "")
    print(f"EXPECTED        : {expected!r}")
    print(f"MODEL EXTRACTED : {result['extracted']!r}")
    if expected:
        print(f"CORRECT         : {is_correct_local(result['extracted'], expected)}")
    print(f"TOKENS / STEPS  : {result['new_tokens']} tok / {result['step_count']} steps / {result['seconds']:.1f}s")
    print("-" * 78)
    print("OUTPUT (post-</think>):")
    print(result["answer_tail"][:400] or "(empty — </think> never closed)")
    print("-- full response excerpt --")
    print(result["response"][:800].rstrip() + ("..." if len(result["response"]) > 800 else ""))
    print()


print("Generation helpers loaded.")

## 10 · Run causal SFT adapter

In [ ]:
causal_results = []
for i, ex in enumerate(SAMPLES, 1):
    r = generate_one(causal_model, tokenizer, ex["question"], ex.get("dataset", "external"))
    causal_results.append((ex, r))
    show_result(i, ex, r, "SFT CAUSAL (Qwen3-1.7B + causal LoRA)")

causal_correct = sum(is_correct_local(r["extracted"], e["answer"]) for e, r in causal_results if e.get("answer"))
causal_graded  = sum(1 for e, _ in causal_results if e.get("answer"))
print(f"\nSFT CAUSAL accuracy: {causal_correct}/{causal_graded} = {causal_correct/max(causal_graded,1)*100:.1f}%")

## 11 · Run base model (adapter disabled)

`disable_adapter()` hides the LoRA weights so the same model instance acts as the raw base —  
no need to reload Qwen3-1.7B a second time, saving VRAM and time.

In [ ]:
base_results = []
with causal_model.disable_adapter():
    for i, ex in enumerate(SAMPLES, 1):
        r = generate_one(causal_model, tokenizer, ex["question"], ex.get("dataset", "external"))
        base_results.append((ex, r))
        show_result(i, ex, r, "BASE (Qwen3-1.7B, no fine-tuning)")

base_correct = sum(is_correct_local(r["extracted"], e["answer"]) for e, r in base_results if e.get("answer"))
base_graded  = sum(1 for e, _ in base_results if e.get("answer"))
print(f"\nBASE accuracy: {base_correct}/{base_graded} = {base_correct/max(base_graded,1)*100:.1f}%")

## 12 · Load noncausal SFT adapter + run

The noncausal adapter is applied to the **same base model** that was already loaded.  
We free the causal adapter first, then load the noncausal one.

In [ ]:
# Free causal adapter, keep the underlying base
del causal_model
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM after freeing causal adapter: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

print(f"Applying noncausal SFT adapter: {NONCAUSAL_ADAPTER_PATH}")
noncausal_model = PeftModel.from_pretrained(_base, NONCAUSAL_ADAPTER_PATH)
noncausal_model.eval()
print(f"  VRAM after noncausal adapter: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

noncausal_results = []
for i, ex in enumerate(SAMPLES, 1):
    r = generate_one(noncausal_model, tokenizer, ex["question"], ex.get("dataset", "external"))
    noncausal_results.append((ex, r))
    show_result(i, ex, r, "SFT NONCAUSAL (Qwen3-1.7B + noncausal LoRA)")

noncausal_correct = sum(is_correct_local(r["extracted"], e["answer"]) for e, r in noncausal_results if e.get("answer"))
noncausal_graded  = sum(1 for e, _ in noncausal_results if e.get("answer"))
print(f"\nSFT NONCAUSAL accuracy: {noncausal_correct}/{noncausal_graded} = {noncausal_correct/max(noncausal_graded,1)*100:.1f}%")

## 13 · 3-way comparison table: Base vs Causal SFT vs Noncausal SFT

In [ ]:
row_fmt = "{:<3} {:<20} {:<7} {:<7} {:<7} {:<7} {:<7} {:<7} {:<7} {:<7}"
print(row_fmt.format(
    "#", "expected",
    "b_ok", "c_ok", "n_ok",
    "b_tok", "c_tok", "n_tok",
    "b_stp", "c_stp"
))
print("-" * 100)

tok_b = tok_c = tok_n = stp_b = stp_c = stp_n = 0
for i, ((eb, rb), (ec, rc), (en, rn)) in enumerate(
        zip(base_results, causal_results, noncausal_results), 1):
    bok = is_correct_local(rb["extracted"], eb.get("answer", "")) if eb.get("answer") else False
    cok = is_correct_local(rc["extracted"], ec.get("answer", "")) if ec.get("answer") else False
    nok = is_correct_local(rn["extracted"], en.get("answer", "")) if en.get("answer") else False
    tok_b += rb["new_tokens"]; tok_c += rc["new_tokens"]; tok_n += rn["new_tokens"]
    stp_b += rb["step_count"]; stp_c += rc["step_count"]; stp_n += rn["step_count"]
    print(row_fmt.format(
        i, str(eb.get("answer", ""))[:19],
        "Y" if bok else "N",
        "Y" if cok else "N",
        "Y" if nok else "N",
        rb["new_tokens"], rc["new_tokens"], rn["new_tokens"],
        rb["step_count"], rc["step_count"],
    ))

n = max(len(SAMPLES), 1)
print("-" * 100)
print(f"BASE      accuracy: {base_correct}/{base_graded} = {base_correct/max(base_graded,1)*100:.1f}%  "
      f"avg tokens: {tok_b/n:.0f}  avg steps: {stp_b/n:.1f}")
print(f"CAUSAL    accuracy: {causal_correct}/{causal_graded} = {causal_correct/max(causal_graded,1)*100:.1f}%  "
      f"avg tokens: {tok_c/n:.0f}  avg steps: {stp_c/n:.1f}")
print(f"NONCAUSAL accuracy: {noncausal_correct}/{noncausal_graded} = {noncausal_correct/max(noncausal_graded,1)*100:.1f}%  "
      f"avg tokens: {tok_n/n:.0f}  avg steps: {stp_n/n:.1f}")
print()
print(f"Causal vs Noncausal: {(tok_c-tok_n)/n:+.0f} tokens avg  ({(tok_c-tok_n)/max(tok_n,1)*100:+.1f}%)")
print()
print("Expected: Causal ≈ Noncausal in accuracy, but Causal uses significantly fewer tokens")
print("because it learned PNS-pruned chains (only causally necessary reasoning steps).")

## 14 · External test data (CSV / JSONL)

If the instructor uploads their own test file, set `EXTERNAL_PATH` and run this cell.

**Supported schemas:**  
- **JSONL** — keys: `question` (or `problem` / `prompt`), `answer` (or `ground_truth` / `answerKey`), optionally `dataset`.  
- **CSV** — same column names. Without `answer`, results print without correctness flag.

In [ ]:
QUESTION_KEYS = ("question", "problem", "prompt")
ANSWER_KEYS   = ("answer", "ground_truth", "answerKey")


def _pick(record, keys):
    for k in keys:
        if k in record and record[k] not in (None, ""):
            return record[k]
    return None


def load_external(path):
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    if path.lower().endswith(".jsonl"):
        with open(path, encoding="utf-8") as f:
            rows = [json.loads(line) for line in f if line.strip()]
    elif path.lower().endswith(".csv"):
        import csv
        with open(path, encoding="utf-8", newline="") as f:
            rows = list(csv.DictReader(f))
    else:
        raise ValueError(f"Unsupported extension: {path} (need .jsonl or .csv)")
    examples = []
    for r in rows:
        q = _pick(r, QUESTION_KEYS)
        if not q:
            continue
        examples.append({
            "question": str(q),
            "answer":   str(_pick(r, ANSWER_KEYS) or ""),
            "dataset":  str(r.get("dataset", "external")).lower(),
        })
    return examples


def run_external(path, model, tok, model_label="SFT", limit=None):
    examples = load_external(path)
    if limit:
        examples = examples[:limit]
    print(f"Loaded {len(examples)} examples from {path}")
    out = []
    correct = n_graded = 0
    for i, ex in enumerate(examples, 1):
        r = generate_one(model, tok, ex["question"], ex["dataset"])
        show_result(i, ex, r, model_label)
        out.append({**ex, **r})
        if ex["answer"]:
            n_graded += 1
            correct  += int(is_correct_local(r["extracted"], ex["answer"]))
    if n_graded:
        print(f"\nAccuracy: {correct}/{n_graded} = {correct/n_graded*100:.1f}%")
    else:
        print("\nNo ground-truth answers — accuracy not computed.")
    return out


# Uncomment to run on instructor's test file:
# EXTERNAL_PATH = "/content/drive/MyDrive/instructor_test.jsonl"   # or .csv
# external_outputs = run_external(EXTERNAL_PATH, noncausal_model, tokenizer, model_label="SFT NONCAUSAL", limit=20)

### Optional — save all results and download

In [ ]:
# OUTPUT_PATH = "sft_inference_results.jsonl"
# with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
#     for (eb, rb), (ec, rc), (en, rn) in zip(base_results, causal_results, noncausal_results):
#         f.write(json.dumps({
#             "question":              eb["question"],
#             "ground_truth":          eb.get("answer", ""),
#             "dataset":               eb.get("dataset", "gsm8k"),
#             "base_extracted":        rb["extracted"],
#             "causal_extracted":      rc["extracted"],
#             "noncausal_extracted":   rn["extracted"],
#             "base_correct":          is_correct_local(rb["extracted"], eb.get("answer", "")),
#             "causal_correct":        is_correct_local(rc["extracted"], ec.get("answer", "")),
#             "noncausal_correct":     is_correct_local(rn["extracted"], en.get("answer", "")),
#             "base_tokens":           rb["new_tokens"],
#             "causal_tokens":         rc["new_tokens"],
#             "noncausal_tokens":      rn["new_tokens"],
#             "base_response":         rb["response"],
#             "causal_response":       rc["response"],
#             "noncausal_response":    rn["response"],
#         }, ensure_ascii=False) + "\n")
# from google.colab import files
# files.download(OUTPUT_PATH)